In [ ]:
r"""
# 05. Authoritative Qualitative Case Study Analysis (G0 vs G3)

### Experimental Provenance
- **Candidate Pool**: Exactly 4,278 candidate drugs from PrimeKG
- **Graph Variants**: G0 (DDI backbone only) vs. G3 (full biomedical context)
- **G3 Ranking**: Computed in this notebook from the preserved G3 seed-44 lightweight runtime.
- **G0 Ranking**: Reproduced twice from the recovered original G0 seed-44 checkpoint and G0 graph using the validated filtered-ranking evaluator. The verified directional ranks are preserved in `results/case_study_three_pair/G0_seed44_reproduction_audit.txt` and entered below.
- **Scoring Decoder**: DistMult decoder scoring $s(u, v) = u \cdot (v \odot r_{\text{ddi}})$
- **Ranking Protocol**: All 4,278 candidates; known-positive and self filtering; evaluated target restored; strict rank = 1 + number of candidates scoring higher; forward and reverse directions evaluated.
- **Biomedical Context**: PrimeKG G3 context from `g3_drug_context.csv`.
- **Case-study status**: Acetylsalicylic acid-Ibuprofen is a known-positive pair in the complete mask; Colchicine-Probenecid and Metformin-Glipizide are unobserved candidate links. These ranks are research-model outputs, not clinical evidence.
"""

import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = Path("..") if Path("..").resolve().name.startswith("CHEERS") else Path(".")
if str(BASE_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(BASE_DIR.resolve()))

LIGHTWEIGHT_DIR = BASE_DIR / "final_release" / "lightweight_runtime"

runtime_g3 = np.load(LIGHTWEIGHT_DIR / "ddi_runtime_embeddings.npz")
emb_g3 = runtime_g3["candidate_embeddings"]
rel_g3 = runtime_g3["ddi_relation"]

mask_data = np.load(LIGHTWEIGHT_DIR / "known_positive_mask_packed.npz")
key_name = list(mask_data.keys())[0]
known_mask = np.unpackbits(mask_data[key_name], axis=1)[:, :len(emb_g3)].astype(bool)

drug_meta = pd.read_csv(LIGHTWEIGHT_DIR / "drug_metadata.csv")
lookup = {}
for row_idx, (_, row) in enumerate(drug_meta.iterrows()):
    for val in row.values:
        if pd.notna(val):
            lookup[str(val).strip()] = row_idx
            lookup[str(val).strip().lower()] = row_idx

def get_cand_idx(ident):
    return lookup.get(str(ident).strip()) or lookup.get(str(ident).strip().lower())

def evaluate_pair_rank(emb, rel, head_id, tail_id):
    h_idx, t_idx = get_cand_idx(head_id), get_cand_idx(tail_id)
    
    s_fwd = emb[h_idx] @ (emb * rel).T
    m_fwd = known_mask[h_idx].copy()
    m_fwd[h_idx] = True; m_fwd[t_idx] = False
    s_fwd[m_fwd] = -np.inf
    rank_fwd = int(np.sum(s_fwd > s_fwd[t_idx])) + 1
    
    s_rev = emb[t_idx] @ (emb * rel).T
    m_rev = known_mask[t_idx].copy()
    m_rev[t_idx] = True; m_rev[h_idx] = False
    s_rev[m_rev] = -np.inf
    rank_rev = int(np.sum(s_rev > s_rev[h_idx])) + 1
    
    return rank_fwd, rank_rev, min(rank_fwd, rank_rev)

print(f"Loaded {len(emb_g3)} candidate drugs. G3 ranking engine initialized.")

test_pairs = [
    {"drug_a": "DB01394", "name_a": "Colchicine", "drug_b": "DB01032", "name_b": "Probenecid", "g0_fwd": 1, "g0_rev": 2},
    {"drug_a": "DB00945", "name_a": "Acetylsalicylic acid", "drug_b": "DB01050", "name_b": "Ibuprofen", "g0_fwd": 2, "g0_rev": 2},
    {"drug_a": "DB00331", "name_a": "Metformin", "drug_b": "DB01067", "name_b": "Glipizide", "g0_fwd": 5, "g0_rev": 1}
]

rows = []
for p in test_pairs:
    g3_fwd, g3_rev, g3_best = evaluate_pair_rank(emb_g3, rel_g3, p["drug_a"], p["drug_b"])
    g0_best = min(p["g0_fwd"], p["g0_rev"])
    
    rank_diff = g0_best - g3_best

    if rank_diff >= 5:
        category = "Improved"
    elif rank_diff <= -5:
        category = "Degraded"
    else:
        category = "Neutral"

    rows.append({
        "Category": category,
        "Drug A": p["name_a"],
        "Drug B": p["name_b"],
        "DrugBank A": p["drug_a"],
        "DrugBank B": p["drug_b"],
        "G0 Fwd": p["g0_fwd"],
        "G0 Rev": p["g0_rev"],
        "G0 Best": g0_best,
        "G3 Fwd": g3_fwd,
        "G3 Rev": g3_rev,
        "G3 Best": g3_best
    })

df_case = pd.DataFrame(rows)
res_path = BASE_DIR / "results" / "case_study_three_pair" / "case_study_ranks.csv"
res_path.parent.mkdir(parents=True, exist_ok=True)
df_case.to_csv(res_path, index=False)
display(df_case)

plt.figure(figsize=(8, 5))
c_map = {"Improved": "#2ca02c", "Neutral": "#7f7f7f", "Degraded": "#d62728"}

for _, r in df_case.iterrows():
    plt.plot([0, 1], [r["G0 Best"], r["G3 Best"]], marker='o', linewidth=2.5, color=c_map[r["Category"]], label=f"{r['Drug A']} – {r['Drug B']} ({r['Category']})")

plt.xticks([0, 1], ["G0 (DDI Baseline)", "G3 (Full Context)"], fontsize=11, fontweight='bold')
plt.gca().invert_yaxis()
plt.ylabel("Filtered Rank (Lower is Better)", fontsize=11)
plt.title("Qualitative Case Study: Rank Progression Across Graph Variants", fontsize=12, fontweight='bold')
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(frameon=True)
plt.tight_layout()

fig_path = BASE_DIR / "figures" / "case_study_three_pair" / "case_study_rank_chart.png"
fig_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(fig_path, dpi=300)
plt.show()

ctx_file = BASE_DIR / "final_release" / "g3_context_runtime" / "g3_drug_context.csv"
df_ctx = pd.read_csv(ctx_file)

cols = {c.lower(): c for c in df_ctx.columns}
id_c = cols["drug_id"]
ent_c = cols["context_name"]
grp_c = cols["context_group"]
rel_c = cols["relation"]

ctx_a = df_ctx[df_ctx[id_c] == "DB01394"]
ctx_b = df_ctx[df_ctx[id_c] == "DB01032"]
shared = set(ctx_a[ent_c]).intersection(set(ctx_b[ent_c]))

print("=== Supporting Biomedical Graph Context (Colchicine & Probenecid) ===")
print(f"Total Shared Entities available in G3: {len(shared)}")
for ent in list(shared)[:5]:
    grp = ctx_a[ctx_a[ent_c] == ent][grp_c].iloc[0]
    ra = list(ctx_a[ctx_a[ent_c] == ent][rel_c].unique())
    rb = list(ctx_b[ctx_b[ent_c] == ent][rel_c].unique())
    print(f"- [{str(grp).upper()}] {ent}")
    print(f"    Drug A associations: {ra}")
    print(f"    Drug B associations: {rb}")